# Compose a Granite Switch checkpoint

**What you'll learn.** By the end of this tutorial you'll know how to compose a base Granite model with one or more LoRA adapter libraries into your own Granite Switch checkpoint — a single artifact you can serve with vLLM and drive from mellea, just like the pre-built checkpoints used elsewhere in this repo. You'll also see how to slim the build down by including or excluding specific adapters and by picking between `alora` and `lora` flavors.

**Time required.** ~10–20 minutes end-to-end on a fast connection, most of it spent in §3 downloading the base model and adapters on the first run. Subsequent runs hit the HF cache and finish in under a minute. §5 (serving + generation) needs a GPU; the rest runs on CPU.

Other tutorials in this folder (e.g. [`hello_mellea.ipynb`](../quickstart/hello_mellea.ipynb), [`02_govt_rag_pipeline.ipynb`](02_govt_rag_pipeline.ipynb)) **consume** a pre-composed Granite Switch model — a single checkpoint with multiple LoRA adapters baked in, served by vLLM and driven through mellea intrinsics. This notebook shows the other half: **how to produce such a checkpoint yourself**, from a base Granite model plus one or more adapter libraries.

After running §1–§3 below you'll have a local directory (`./granite-switch-tutorials/`) containing a checkpoint that powers every intrinsic those two notebooks call: `guardian_check`, `policy_guardrails`, `factuality_detection`, `factuality_correction`, `rewrite_question`, `check_answerability`, `clarify_query`, `find_citations`, `flag_hallucinated_content`, and `check_context_relevance`.

Execute the notebook top-to-bottom — only §2 and §3 actually do work by default — or read it as a recipe book: §4 demonstrates selection flags (`--include-adapters`, `--exclude-adapters`, `--technology-filter`) you can mix in, and those cells are pre-commented so re-running the notebook doesn't rebuild multiple checkpoints. §5 shows how to serve the result, and §6 runs a tiny generation against it. For the canonical CLI reference see [`src/granite_switch/composer/README.md`](../../src/granite_switch/composer/README.md).

**Prerequisites.** If any adapter library is gated on Hugging Face, run `huggingface-cli login` first. A full build of all three libraries in §3 downloads the base model (~6 GB for the 3B bf16 `granite-4.1-3b`) plus adapters, and writes the composed checkpoint to disk — plan for ~20 GB free space and several minutes on a fast connection.

## 1 · Configuration

Edit these if you want a different base model, different libraries, or a different output directory.

In [ ]:
BASE_MODEL   = "ibm-granite/granite-4.1-3b"
OUTPUT_DIR   = "./granite-switch-tutorials"

RAG_LIB      = "ibm-granite/granitelib-rag-r1.0"       # query_rewrite, answerability, citations, ...
CORE_LIB     = "ibm-granite/granitelib-core-r1.0"      # requirement-check, uncertainty, context-attribution
GUARDIAN_LIB = "ibm-granite/granitelib-guardian-r1.0"  # guardian-core, policy-guardrails, factuality-*

print(f"base:    {BASE_MODEL}")
print(f"output:  {OUTPUT_DIR}")
print(f"libs:    {RAG_LIB}\n         {CORE_LIB}\n         {GUARDIAN_LIB}")

## 2 · Preview what's available — `--list-adapters`

Ask the composer what each library contains. `--list-adapters` prints the adapter inventory and exits without writing anything — useful for deciding what to include before committing to a full build. When both `alora` and `lora` flavors exist for the same adapter, the composer prefers `alora` by default.

> **Note:** This step is optional — it only previews the adapters in each library and doesn't affect the build in §3. If the cell fails (e.g. `unrecognized arguments: --list-adapters`) it usually means the `granite_switch` package on your notebook kernel's Python path is older than this repo. You can safely skip ahead to §3; the compose step doesn't need `--list-adapters`. To fix it, reinstall the package in the kernel's environment (`pip install "granite-switch[compose]"`) and restart the kernel.

In [ ]:
!python -m granite_switch.composer.compose_granite_switch \
  --base-model {BASE_MODEL} \
  --adapters {RAG_LIB} {CORE_LIB} {GUARDIAN_LIB} \
  --list-adapters

## 3 · Compose the model

Pull all adapters from all three libraries and embed them into the base model. Takes a few minutes and downloads ~15 GB (base model + adapters) on the first run; subsequent runs hit the HF cache.

In [ ]:
!python -m granite_switch.composer.compose_granite_switch \
  --base-model {BASE_MODEL} \
  --adapters {RAG_LIB} {CORE_LIB} {GUARDIAN_LIB} \
  --output {OUTPUT_DIR}

Two files in the output directory are worth looking at. **`BUILD.md`** is a human-readable summary — the adapter table in it tells you the control token (e.g. `<|answerability|>`) that mellea will route intrinsic calls through. **`adapter_index.json`** is the same mapping in machine-readable form, used at inference time.

In [ ]:
!ls {OUTPUT_DIR}
print("\n--- first 60 lines of BUILD.md ---")
!head -60 {OUTPUT_DIR}/BUILD.md

## 4 · Selecting which adapters to include

By default the composer embeds every adapter it finds in the libraries you point it at. That's a reasonable place to start, but for production you'll often want a leaner checkpoint: fewer embedded adapters means a smaller safetensors file, lower VRAM at serve time, and a faster cold start.

These flags narrow the set:

| Flag | What it does |
|---|---|
| `--include-adapters` | Keep only adapters matching these names or [fnmatch globs](https://docs.python.org/3/library/fnmatch.html) (e.g. `'query_*'`). |
| `--exclude-adapters` | Drop adapters matching these names/globs. Applied *after* `--include-adapters`. |
| `--technology-filter` | Force `alora` or `lora` when both exist for an adapter (default: prefer `alora`). |

The two cells below are pre-commented; uncomment one, change `--output`, and run it to see the effect.

**Example A — `--include-adapters`**: a lean checkpoint with only the intrinsics used in [`hello_mellea.ipynb`](../quickstart/hello_mellea.ipynb) (guardian + 4 RAG adapters).

In [ ]:
# !python -m granite_switch.composer.compose_granite_switch \
#   --base-model {BASE_MODEL} \
#   --adapters {RAG_LIB} {GUARDIAN_LIB} \
#   --include-adapters guardian-core query_rewrite answerability \
#                      query_clarification citations \
#   --output ./granite-switch-hello-world

**Example B — `--exclude-adapters` + `--technology-filter`**: everything *except* the factuality adapters, using the LoRA flavor where both exist.

In [ ]:
# !python -m granite_switch.composer.compose_granite_switch \
#   --base-model {BASE_MODEL} \
#   --adapters {RAG_LIB} {CORE_LIB} {GUARDIAN_LIB} \
#   --exclude-adapters 'factuality-*' \
#   --technology-filter lora \
#   --output ./granite-switch-lora-no-factuality

## 5 · Generate against the composed model

Start a vLLM server pointing at the directory §3 produced:

```bash
python -m vllm.entrypoints.openai.api_server \
  --model ./granite-switch-tutorials \
  --port 8000
```

Then run the cell below — it connects mellea to the server, registers the embedded adapters, and calls the `query_rewrite` intrinsic. If it prints a cleaned-up version of the messy query, your composed checkpoint is wired up correctly.

In [ ]:
from mellea.backends.openai import OpenAIBackend
from mellea.stdlib.context import ChatContext
from mellea.stdlib.components.intrinsic import rag

backend = OpenAIBackend(
    model_id=OUTPUT_DIR,
    base_url="http://localhost:8000/v1",
    api_key="unused",
)
backend.register_embedded_adapter_model(OUTPUT_DIR)
print(f"Adapters: {backend.list_adapters()}")

query = "I want to ask you something. what is...mmmm the the main city(capital you call it,right?) of France?"
rewritten = rag.rewrite_question(query, ChatContext(), backend)
print(f"\noriginal:  {query}")
print(f"rewritten: {rewritten}")